# Chapter 06: Camera Models

Source orientation: printed pages 153-177; PDF pages 171-195.

This notebook is an original, standalone computational treatment of the chapter. The PDF was used only to identify the chapter structure, concepts, and algorithmic emphasis. It does not reproduce textbook prose, figures, screenshots, long exercise text, or page crops. The goal is to turn the chapter into an inspectable multiple-view-geometry lab that a reader can use without keeping the book open.

## Chapter Goal

Finite cameras, projective cameras, affine cameras, intrinsic/extrinsic factorization, and alternative camera models.

Multiple-view geometry becomes easier to learn when every algebraic object is paired with a scene, a measurement, and a diagnostic. This notebook therefore treats the chapter as a working vision problem rather than as a sequence of isolated formulas. Points, lines, cameras, conics, tensors, residuals, and optimization variables are represented in executable form. The visuals are designed to reveal what survives a projection, what is lost, which constraints are merely algebraic, and which constraints are geometric.

The examples use deterministic synthetic data: calibrated and uncalibrated cameras, planar grids, cube or room-like point sets, noisy correspondences, and small track matrices. Synthetic data is intentional because every artifact can be regenerated, perturbed, and checked. Real images are valuable in applications, but the central ideas of this chapter are clearest when the ground truth geometry is known and the failure modes can be turned on deliberately.

## Translation Guide

- **Homogeneous data:** points, lines, cameras, planes, and tensors are represented up to scale, so every equality that involves them must be phrased as a proportionality, an incidence relation, a rank condition, or a normalized residual.
- **Camera action:** a camera is a projective map from 3D scene coordinates to 2D image coordinates. The code always distinguishes the camera center, the image measurement, and the back-projected ray or plane that connects them.
- **Invariant:** the important question is not whether an array changed, but whether the geometric relation survived: collinearity, coplanarity, cross-ratio, rank, epipolar incidence, positive depth, or reprojection error.
- **Estimator:** a linear algorithm supplies an initial model; geometric, statistical, or robust criteria decide whether that model explains the measurements.
- **Artifact:** each figure is saved under the book-local `artifacts/` tree, displayed inline, and checked in the final cell so the notebook remains reproducible.

## Route Through The Chapter

1. Name the geometric object and its computational representation.
2. Build a small scene where the object can be projected, transformed, or estimated.
3. Draw the construction in a way that makes the invariant visible.
4. Perturb the data to expose conditioning, uncertainty, or ambiguity.
5. Close with checks that assert the core relations and artifact integrity.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = 'chapter-06'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


## Visual Storyboard And Library Routing

This chapter uses **NumPy** for the camera-matrix algebra because the key objects are nullspaces, block factorizations, homogeneous scale classes, and projections. **Matplotlib 3D** is used for the finite-camera frustum because the learner needs to see the center, image plane, and rays in one coordinate frame. Plain **Matplotlib 2D** is used for intrinsic grids and projection-class comparisons because those are image-plane measurements, not decorative scenes. The course artifact helpers save every visual under `artifacts/chapter-06` so the notebook remains auditable without the PDF.

| Visual | Artifact | What to inspect | Check |
| --- | --- | --- | --- |
| Camera-matrix factorization map | `figures/camera-matrix-factorization-map.png` | how `K`, `R`, and `C` compose the finite camera matrix | scaled reconstruction of `P` |
| Frustum and nullspace scene | `figures/finite-camera-frustum-nullspace.png` | the center is behind every projected ray and satisfies `P C = 0` | camera-center nullspace residual |
| Intrinsic-parameter grid | `figures/intrinsic-parameter-grid.png` | focal length, skew, and principal point change image coordinates but not the 3D ray idea | homogeneous scale invariance |
| Perspective versus affine projection | `figures/perspective-affine-projection-comparison.png` | finite perspective can make parallel rails meet, while affine projection keeps their image directions parallel | affine direction cross-product residual |

## Core Concepts

### 1. A camera matrix maps world rays to image points through homogeneous projection

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **camera frustum and image plane**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **P times the camera center is zero**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 2. The camera center is the nullspace of p

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **intrinsic parameter slider map**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **RQ-style factorization reconstructs the camera matrix**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 3. Intrinsics and extrinsics separate internal pixel geometry from camera pose

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **projection-class comparison**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **projected homogeneous coordinates are scale equivalent**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 4. Affine and weak-perspective cameras are controlled approximations

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **camera-center nullspace diagram**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **affine projection keeps parallel world directions parallel in the image**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

## Worked Example Pattern

The worked examples use a shared synthetic lab. A few cameras view a simple 3D scene, those cameras produce image measurements, and the chapter-specific object is computed from the measurements. This pattern is repeated because it makes the course cumulative: homographies from Part 0 return as plane-induced transfers in Part II, camera matrices from Part I feed epipolar geometry, and two-view triangulation becomes a building block for N-view bundle adjustment.

For this chapter, the important work is to connect **Finite cameras, projective cameras, affine cameras, intrinsic/extrinsic factorization, and alternative camera models.** to observable behavior. The first figure is a concept map that states what to watch. The second figure is a geometry scene. The third figure is a diagnostic view where residuals, conditioning, or covariance can be inspected. The fourth figure is a rank or constraint dashboard, because many multiple-view failures announce themselves as a singular value that should not be ignored.

The code is intentionally compact. It is not a production vision library; it is a transparent teaching implementation that exposes each step. The reusable helpers live in `utils/` so later chapters can use the same projection, epipolar, estimation, and plotting vocabulary.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import rq

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib
from utils.cameras import camera_center, camera_matrix, cube_points, look_at_rotation, make_calibration, project_points
from utils.projective import dehomogenize, homogenize

ENTRY_TITLE = 'Camera Models'
MODE = 'camera'
TOPIC = 'chapter-06'
CONCEPTS = ['finite camera matrix', 'camera-center nullspace', 'intrinsic calibration', 'extrinsic pose', 'affine and weak-perspective projection']
VISUALS = ['camera-matrix factorization map', 'finite-camera frustum and nullspace', 'intrinsic-parameter image grid', 'perspective-affine projection comparison']
CHECKS = ['P times the camera center is zero', 'K[R|-RC] reconstructs the finite camera matrix up to scale', 'projected homogeneous coordinates are scale equivalent', 'affine projection keeps parallel world directions parallel in the image']
SEED = 107
artifact_paths = []

K = make_calibration(850.0, 830.0, 320.0, 240.0, skew_value=18.0)
C = np.array([-1.8, 0.45, -4.15])
R = look_at_rotation(C, target=np.array([0.0, 0.05, 3.2]))
P = camera_matrix(K, R, C)
C_h = camera_center(P)
C_euclid = C_h[:3] / C_h[3]
points3d = cube_points(scale=0.62)
image_points = project_points(P, points3d)
normalized_points = dehomogenize((R @ (points3d - C).T).T)

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 4.8))
ax.axis('off')
ax.set_title('Finite camera matrix as three inspectable choices', loc='left', fontsize=13, fontweight='bold')
blocks = [
    ('K', 'intrinsics/nfocal lengths, skew, principal point', (0.08, 0.57), '#2563eb'),
    ('R', 'orientation/nworld axes in camera coordinates', (0.34, 0.57), '#0f766e'),
    ('C', 'camera center/nnull vector of P', (0.60, 0.57), '#c2410c'),
    ('P = K [R | -RC]', 'projective camera/n3D homogeneous point -> image point', (0.34, 0.18), '#6d28d9'),
]
for label, detail, (x, y), color in blocks:
    rect = plt.Rectangle((x, y), 0.22, 0.18, facecolor=color, edgecolor='white', lw=1.8, alpha=0.94)
    ax.add_patch(rect)
    ax.text(x + 0.11, y + 0.115, label, color='white', ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(x + 0.11, y + 0.035, detail, color='white', ha='center', va='center', fontsize=7.8)
for start, end in [((0.30, 0.66), (0.34, 0.66)), ((0.56, 0.66), (0.60, 0.66)), ((0.45, 0.57), (0.45, 0.36))]:
    ax.annotate('', xy=end, xytext=start, arrowprops=dict(arrowstyle='->', lw=2.0, color='#334155'))
ax.text(0.08, 0.08, 'Inspection target: each camera model changes one block while preserving homogeneous projection logic.', fontsize=9.5, color='#1f2937')
ax.text(0.08, 0.025, 'Camera models in the chapter differ by which blocks are finite, calibrated, restricted, or approximated.', fontsize=9.2, color='#475569')
factorization_path = save_matplotlib(fig, TOPIC, 'figures', 'camera-matrix-factorization-map.png')
plt.close(fig)
artifact_paths.append(factorization_path)
display_artifact(factorization_path, width=860)

In [ ]:
fig = plt.figure(figsize=(8.8, 6.6))
ax = fig.add_subplot(111, projection='3d')
ax.set_title('Finite camera frustum: center, image plane, and projected rays', loc='left', fontsize=12, fontweight='bold')
ax.scatter(points3d[:, 0], points3d[:, 1], points3d[:, 2], s=34, c='#0f766e', depthshade=True, label='3D cube points')
ax.scatter([C_euclid[0]], [C_euclid[1]], [C_euclid[2]], s=90, marker='^', c='#c2410c', label='camera center C')
for X in points3d[:8]:
    ax.plot([C_euclid[0], X[0]], [C_euclid[1], X[1]], [C_euclid[2], X[2]], color='#94a3b8', lw=0.9, alpha=0.8)
plane_center = C_euclid + R.T @ np.array([0.0, 0.0, 1.35])
plane_u = R.T @ np.array([0.75, 0.0, 0.0])
plane_v = R.T @ np.array([0.0, 0.55, 0.0])
corners = np.array([plane_center - plane_u - plane_v, plane_center + plane_u - plane_v, plane_center + plane_u + plane_v, plane_center - plane_u + plane_v])
for a, b in zip(corners, np.roll(corners, -1, axis=0)):
    ax.plot([a[0], b[0]], [a[1], b[1]], [a[2], b[2]], color='#2563eb', lw=2.0)
ax.text(C_euclid[0], C_euclid[1], C_euclid[2], '  PC = 0', color='#c2410c')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.view_init(elev=22, azim=-54)
ax.legend(loc='upper left', fontsize=8)
frustum_path = save_matplotlib(fig, TOPIC, 'figures', 'finite-camera-frustum-nullspace.png')
plt.close(fig)
artifact_paths.append(frustum_path)
display_artifact(frustum_path, width=840)

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-0.38, 0.38, 5), np.linspace(-0.28, 0.28, 5))
rays = np.column_stack([grid_x.ravel(), grid_y.ravel(), np.ones(grid_x.size)])
intrinsic_cases = [
    ('baseline K', K, '#2563eb'),
    ('longer focal length', make_calibration(1120.0, 1090.0, 320.0, 240.0, skew_value=18.0), '#0f766e'),
    ('skew + shifted principal point', make_calibration(850.0, 830.0, 390.0, 205.0, skew_value=95.0), '#c2410c'),
]
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.8), sharex=True, sharey=True)
for ax, (label, K_case, color) in zip(axes, intrinsic_cases):
    pix = dehomogenize((K_case @ rays.T).T)
    ax.scatter(pix[:, 0], pix[:, 1], s=30, c=color)
    for row in range(5):
        part = pix[row * 5:(row + 1) * 5]
        ax.plot(part[:, 0], part[:, 1], color=color, alpha=0.75, lw=1.3)
    for col in range(5):
        part = pix[col::5]
        ax.plot(part[:, 0], part[:, 1], color=color, alpha=0.75, lw=1.3)
    ax.scatter([K_case[0, 2]], [K_case[1, 2]], marker='+', s=120, c='#111827', label='principal point')
    ax.set_title(label, loc='left', fontsize=10, fontweight='bold')
    ax.grid(True, color='#e5e7eb')
    ax.set_aspect('equal', adjustable='box')
    ax.invert_yaxis()
axes[0].set_ylabel('image v coordinate')
for ax in axes:
    ax.set_xlabel('image u coordinate')
fig.suptitle('Intrinsic parameters act on normalized rays before pixel coordinates are measured', x=0.02, ha='left', fontsize=12, fontweight='bold')
fig.tight_layout()
intrinsic_path = save_matplotlib(fig, TOPIC, 'figures', 'intrinsic-parameter-grid.png')
plt.close(fig)
artifact_paths.append(intrinsic_path)
display_artifact(intrinsic_path, width=900)

In [ ]:
def project_perspective_simple(X):
    X = np.asarray(X, dtype=float)
    return X[:, :2] / X[:, 2:3]

def project_affine_simple(X):
    X = np.asarray(X, dtype=float)
    return X[:, :2]

rail_a = np.array([[-0.9, -0.32, 1.8], [-0.45, -0.32, 2.5], [0.00, -0.32, 3.2], [0.45, -0.32, 3.9], [0.90, -0.32, 4.6]])
rail_b = rail_a + np.array([0.0, 0.64, 0.0])
persp_a, persp_b = project_perspective_simple(rail_a), project_perspective_simple(rail_b)
aff_a, aff_b = project_affine_simple(rail_a), project_affine_simple(rail_b)
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.2))
for ax, a_pts, b_pts, title in [(axes[0], persp_a, persp_b, 'finite perspective camera'), (axes[1], aff_a, aff_b, 'affine / weak-perspective camera')]:
    ax.plot(a_pts[:, 0], a_pts[:, 1], '-o', color='#2563eb', label='rail A')
    ax.plot(b_pts[:, 0], b_pts[:, 1], '-o', color='#c2410c', label='rail B')
    for p0, p1 in zip(a_pts, b_pts):
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color='#94a3b8', lw=0.8)
    ax.set_title(title, loc='left', fontsize=10, fontweight='bold')
    ax.grid(True, color='#e5e7eb')
    ax.set_aspect('equal', adjustable='box')
    ax.legend(fontsize=8)
axes[0].set_ylabel('image y')
for ax in axes:
    ax.set_xlabel('image x')
fig.suptitle('Projection class changes which Euclidean properties survive in the image', x=0.02, ha='left', fontsize=12, fontweight='bold')
fig.tight_layout()
projection_class_path = save_matplotlib(fig, TOPIC, 'figures', 'perspective-affine-projection-comparison.png')
plt.close(fig)
artifact_paths.append(projection_class_path)
display_artifact(projection_class_path, width=860)

## Computational Lab

The lab below uses the same checks as the visual storyboard:

- `P times the camera center is zero`
- `RQ-style factorization reconstructs the camera matrix`
- `projected homogeneous coordinates are scale equivalent`
- `affine projection keeps parallel world directions parallel in the image`

The exact values are less important than the workflow. Build a synthetic configuration, compute the projective or statistical object, and verify the invariant that the chapter says should hold. In a real application the data would be image correspondences or tracked features. In this course the data is deterministic so the result can be audited.

The miniature experiment uses two cameras and a cube-like point cloud whenever possible. Even chapters about statistics, tensor notation, or optimization keep the same projective heartbeat: measurements are generated by projection, a model is estimated, and the model is judged by residuals, rank, or covariance. This continuity helps prevent a common misconception in multiple-view geometry: that the algebra and the geometry are separate topics. They are two views of the same constraints.

In [ ]:
# Camera-specific numerical checks, aligned with the four visuals above.
center_null_residual = float(np.linalg.norm(P @ C_h))
P_from_blocks = K @ np.hstack([R, -R @ C.reshape(3, 1)])
scale = float(np.vdot(P, P_from_blocks) / np.vdot(P_from_blocks, P_from_blocks))
block_reconstruction_error = float(np.linalg.norm(P - scale * P_from_blocks) / np.linalg.norm(P))

M = P[:, :3]
K_rq, R_rq = rq(M)
signs = np.sign(np.diag(K_rq))
signs[signs == 0] = 1.0
T = np.diag(signs)
K_rq = K_rq @ T
R_rq = T @ R_rq
if np.linalg.det(R_rq) < 0:
    K_rq[:, 0] *= -1
    R_rq[0, :] *= -1
K_rq = K_rq / K_rq[2, 2]
P_rq = K_rq @ np.hstack([R_rq, -R_rq @ C_euclid.reshape(3, 1)])
rq_scale = float(np.vdot(P, P_rq) / np.vdot(P_rq, P_rq))
rq_reconstruction_error = float(np.linalg.norm(P - rq_scale * P_rq) / np.linalg.norm(P))

X_h = np.array([0.42, -0.18, 3.6, 1.0])
projected_once = dehomogenize((P @ X_h).reshape(1, 3))[0]
projected_scaled = dehomogenize((P @ (7.5 * X_h)).reshape(1, 3))[0]
homogeneous_scale_error = float(np.linalg.norm(projected_once - projected_scaled))

affine_dir_a = aff_a[-1] - aff_a[0]
affine_dir_b = aff_b[-1] - aff_b[0]
perspective_dir_a = persp_a[-1] - persp_a[0]
perspective_dir_b = persp_b[-1] - persp_b[0]
affine_parallel_cross = float(abs(np.cross(affine_dir_a, affine_dir_b)))
perspective_parallel_cross = float(abs(np.cross(perspective_dir_a, perspective_dir_b)))

camera_checks = {
    'source_span': {'printed_pages': '153-177', 'pdf_pages': '171-195'},
    'libraries': ['numpy', 'scipy.linalg.rq', 'matplotlib 2D/3D', 'course camera/projective/artifact helpers'],
    'center_nullspace_residual': center_null_residual,
    'known_block_reconstruction_error': block_reconstruction_error,
    'rq_style_reconstruction_error': rq_reconstruction_error,
    'homogeneous_scale_projection_error': homogeneous_scale_error,
    'affine_parallel_cross_product': affine_parallel_cross,
    'perspective_parallel_cross_product': perspective_parallel_cross,
    'image_point_bounds': {
        'u_min': float(image_points[:, 0].min()),
        'u_max': float(image_points[:, 0].max()),
        'v_min': float(image_points[:, 1].min()),
        'v_max': float(image_points[:, 1].max()),
    },
    'artifact_count': len(artifact_paths),
}
checks_path = save_json(camera_checks, TOPIC, 'checks', 'camera-model-invariants.json')
display_artifact(checks_path)
camera_checks

## Pitfalls And Failure Modes

The main danger in this chapter is to confuse a plausible array with a valid geometric object. A matrix can have the right shape and still violate rank, scale, sign, or incidence constraints. A small algebraic residual can hide a large image-plane error if the coordinate system is poorly normalized. A projective reconstruction can reproject perfectly while still being metrically wrong. A calibration estimate can look numerically precise while being driven by a degenerate camera motion or by points that do not constrain the intended degrees of freedom.

The antidote is to make each assumption executable. When a relation is homogeneous, normalize before comparing. When a model is estimated, inspect both the residual distribution and the singular values. When an upgrade is claimed, state which additional object was fixed: the line at infinity, the plane at infinity, the absolute conic, the absolute dual quadric, or a cheirality condition. When a robust method is used, report inliers and outliers instead of only the final model. These habits keep the notebook honest and make the visualizations diagnostic rather than decorative.

## Applied Lab

Replace the synthetic data in the lab with one of your own small image-measurement sets. Keep the same artifact contract:

1. draw the measurements and the estimated relation;
2. save the figure under `artifacts/chapter-06/figures/`;
3. write a JSON summary under `artifacts/chapter-06/checks/`;
4. assert the invariant that matters for the chapter.

For this notebook, a good extension is to perturb the measurements with three noise levels and compare the resulting diagnostics. Watch whether **P times the camera center is zero** degrades smoothly or fails abruptly. Abrupt failures usually indicate rank loss, degeneracy, a poor parameterization, or an unhandled scale convention.

In [ ]:
final_sanity = {
    'artifact_count': len(artifact_paths),
    'all_artifacts': [str(path.relative_to(BOOK_ROOT)) for path in artifact_paths],
    'check_artifact': str(checks_path.relative_to(BOOK_ROOT)),
    'center_nullspace_residual': center_null_residual,
    'known_block_reconstruction_error': block_reconstruction_error,
    'rq_style_reconstruction_error': rq_reconstruction_error,
    'homogeneous_scale_projection_error': homogeneous_scale_error,
    'affine_parallel_cross_product': affine_parallel_cross,
    'perspective_parallel_cross_product': perspective_parallel_cross,
}
assert_artifacts(artifact_paths, min_bytes=1500)
assert checks_path.exists() and checks_path.stat().st_size > 200
assert final_sanity['artifact_count'] >= 4
assert center_null_residual < 1e-7
assert block_reconstruction_error < 1e-12
assert rq_reconstruction_error < 1e-7
assert homogeneous_scale_error < 1e-10
assert affine_parallel_cross < 1e-12
assert perspective_parallel_cross > 1e-3
final_path = save_json(final_sanity, TOPIC, 'checks', 'final-sanity.json')
final_sanity


## Takeaways

- A camera matrix maps world rays to image points through homogeneous projection.
- The camera center is the nullspace of p.
- Intrinsics and extrinsics separate internal pixel geometry from camera pose.
- Affine and weak-perspective cameras are controlled approximations.

The chapter's durable lesson is that multiple-view geometry is a discipline of representations and invariants. The visual artifacts show the representation; the code checks the invariant; the prose explains why the invariant matters. That triangle is what makes the notebook stand alone from the source text.